# Demo 03: PatchCore on a Bottle-like Anomaly Task

## Learning goals
- show the PatchCore-style workflow in notebook form: nominal memory bank, anomaly map, nearest normal patch, and counterfactual replacement;
- make provenance explicit by showing the retrieved normal patch next to the query patch;
- keep the full logic and graphics inside the notebook.

## Why this demo matters
An anomaly map alone is not a satisfying explanation. What makes PatchCore interesting is the provenance view: which normal patch in the memory bank was used as the comparison target?
## Data source
This notebook is self-contained. The nominal set and the anomaly examples are synthetic bottle-like images generated inside the notebook; no external dataset files are read.



### Notebook display helpers

In [ ]:
import math

import numpy as np
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageFont

try:
    from IPython.display import display
except Exception:
    def display(obj):
        return None

FONT = ImageFont.load_default()


def np_to_pil(array):
    if isinstance(array, Image.Image):
        return array
    if array.dtype != np.uint8:
        array = (np.clip(array, 0.0, 1.0) * 255).astype(np.uint8)
    return Image.fromarray(array)


def titled(image, title, *, width=None):
    base = np_to_pil(image).convert('RGB')
    if width is not None and base.width != width:
        base = base.resize((width, round(base.height * width / base.width)))
    title_height = 22
    panel = Image.new('RGB', (base.width, base.height + title_height), (255, 255, 255))
    panel.paste(base, (0, title_height))
    draw = ImageDraw.Draw(panel)
    draw.rectangle((0, 0, base.width, title_height), fill=(245, 245, 245))
    draw.text((6, 5), title, fill=(20, 20, 20), font=FONT)
    return panel


def montage(panels, *, cols=3, padding=8, bg=(255, 255, 255)):
    prepared = [np_to_pil(panel).convert('RGB') for panel in panels]
    if not prepared:
        return Image.new('RGB', (320, 120), bg)
    cell_w = max(panel.width for panel in prepared)
    cell_h = max(panel.height for panel in prepared)
    rows = math.ceil(len(prepared) / cols)
    canvas = Image.new(
        'RGB',
        (cols * cell_w + (cols + 1) * padding, rows * cell_h + (rows + 1) * padding),
        bg,
    )
    for idx, panel in enumerate(prepared):
        row, col = divmod(idx, cols)
        x = padding + col * (cell_w + padding)
        y = padding + row * (cell_h + padding)
        canvas.paste(panel, (x, y))
    return canvas


def maybe_display(image):
    display(image)
    return image


def draw_box(image_array, box, *, colour=(220, 48, 48), width=3):
    image = np_to_pil(image_array).convert('RGB')
    draw = ImageDraw.Draw(image)
    x, y, w, h = box
    for offset in range(width):
        draw.rectangle((x - offset, y - offset, x + w + offset, y + h + offset), outline=colour)
    return image


def text_panel(lines, title, *, size=(560, 220)):
    image = Image.new('RGB', size, (255, 255, 255))
    draw = ImageDraw.Draw(image)
    draw.text((10, 10), title, fill=(20, 20, 20), font=FONT)
    y = 38
    for line in lines:
        draw.text((12, y), line, fill=(20, 20, 20), font=FONT)
        y += 16
    return image


def bar_chart(values, labels, title, *, colours=None, size=(560, 280), value_fmt='{:.2f}'):
    if colours is None:
        colours = [(42, 157, 143)] * len(values)
    width, height = size
    left, right, top, bottom = 60, 20, 40, 50
    chart_w = width - left - right
    chart_h = height - top - bottom
    image = Image.new('RGB', size, (255, 255, 255))
    draw = ImageDraw.Draw(image)
    draw.text((10, 10), title, fill=(20, 20, 20), font=FONT)
    max_value = max(max(values), 1e-6)
    bar_w = max(12, chart_w // max(len(values) * 2, 2))
    gap = bar_w
    x = left
    baseline = top + chart_h
    draw.line((left, top, left, baseline), fill=(60, 60, 60), width=1)
    draw.line((left, baseline, left + chart_w, baseline), fill=(60, 60, 60), width=1)
    for value, label, colour in zip(values, labels, colours, strict=True):
        bar_h = int((value / max_value) * (chart_h - 10)) if max_value > 0 else 0
        draw.rectangle((x, baseline - bar_h, x + bar_w, baseline), fill=colour)
        draw.text((x, baseline + 6), label[:12], fill=(20, 20, 20), font=FONT)
        draw.text((x, baseline - bar_h - 14), value_fmt.format(value), fill=(20, 20, 20), font=FONT)
        x += bar_w + gap
    return image


def grouped_bar_chart(groups, series_names, matrix, title, *, colours=None, size=(680, 300), value_fmt='{:.2f}'):
    if colours is None:
        colours = [(164, 68, 68), (43, 122, 120), (69, 123, 157)]
    width, height = size
    left, right, top, bottom = 60, 20, 40, 55
    chart_w = width - left - right
    chart_h = height - top - bottom
    image = Image.new('RGB', size, (255, 255, 255))
    draw = ImageDraw.Draw(image)
    draw.text((10, 10), title, fill=(20, 20, 20), font=FONT)
    max_value = max(max(series) for series in matrix)
    baseline = top + chart_h
    draw.line((left, top, left, baseline), fill=(60, 60, 60), width=1)
    draw.line((left, baseline, left + chart_w, baseline), fill=(60, 60, 60), width=1)
    group_w = chart_w // max(len(groups), 1)
    series_count = len(series_names)
    bar_w = max(10, group_w // (series_count + 1))
    for group_idx, group in enumerate(groups):
        base_x = left + group_idx * group_w + 10
        for series_idx, _ in enumerate(series_names):
            value = matrix[series_idx][group_idx]
            bar_h = int((value / max(max_value, 1e-6)) * (chart_h - 12)) if max_value > 0 else 0
            x0 = base_x + series_idx * bar_w
            draw.rectangle((x0, baseline - bar_h, x0 + bar_w - 2, baseline), fill=colours[series_idx % len(colours)])
            draw.text((x0, baseline - bar_h - 12), value_fmt.format(value), fill=(20, 20, 20), font=FONT)
        draw.text((base_x, baseline + 6), group[:12], fill=(20, 20, 20), font=FONT)
    legend_x = width - 210
    for idx, name in enumerate(series_names):
        y = 24 + idx * 16
        draw.rectangle((legend_x, y, legend_x + 10, y + 10), fill=colours[idx % len(colours)])
        draw.text((legend_x + 16, y - 1), name, fill=(20, 20, 20), font=FONT)
    return image

### Patch extraction and PatchCore-style scoring code

In [ ]:
def to_array(image):
    return np.asarray(image.convert('RGB'), dtype=np.float32) / 255.0


def iter_patch_boxes(height, width, patch=16, stride=8):
    boxes = []
    for y in range(0, height - patch + 1, stride):
        for x in range(0, width - patch + 1, stride):
            boxes.append((x, y, patch, patch))
    return boxes


def patch_features(image_array, boxes):
    features = []
    for x, y, w, h in boxes:
        patch = image_array[y:y + h, x:x + w]
        features.append(np.concatenate([patch.mean(axis=(0, 1)), patch.std(axis=(0, 1))]))
    return np.stack(features)


def build_memory_bank(train_images, *, patch=16, stride=8):
    features = []
    metadata = []
    for image_index, image_array in enumerate(train_images):
        boxes = iter_patch_boxes(image_array.shape[0], image_array.shape[1], patch=patch, stride=stride)
        patch_vectors = patch_features(image_array, boxes)
        features.append(patch_vectors)
        for box in boxes:
            metadata.append({'image_index': image_index, 'box': box})
    return {
        'features': np.vstack(features),
        'metadata': metadata,
        'train_images': train_images,
        'patch': patch,
        'stride': stride,
    }


def score_image(image_array, memory_bank, *, top_k=3):
    boxes = iter_patch_boxes(image_array.shape[0], image_array.shape[1], patch=memory_bank['patch'], stride=memory_bank['stride'])
    query_vectors = patch_features(image_array, boxes)
    results = []
    for box, vector in zip(boxes, query_vectors, strict=True):
        distances = np.linalg.norm(memory_bank['features'] - vector[None, :], axis=1)
        nearest_idx = np.argsort(distances)[:top_k]
        results.append({
            'box': box,
            'distance': float(distances[nearest_idx[0]]),
            'nearest': [
                {'distance': float(distances[idx]), 'metadata': memory_bank['metadata'][int(idx)]}
                for idx in nearest_idx
            ],
        })
    results.sort(key=lambda item: item['distance'], reverse=True)
    return results


def heatmap_from_scores(scores, shape):
    heat = np.zeros(shape[:2], dtype=np.float32)
    weight = np.zeros(shape[:2], dtype=np.float32)
    for item in scores:
        x, y, w, h = item['box']
        heat[y:y + h, x:x + w] += item['distance']
        weight[y:y + h, x:x + w] += 1.0
    return heat / np.maximum(weight, 1.0)


def overlay_heatmap(image_array, heatmap):
    norm = heatmap / max(float(heatmap.max()), 1e-6)
    overlay = image_array.copy()
    overlay[:, :, 0] = np.clip(overlay[:, :, 0] + 0.75 * norm, 0.0, 1.0)
    overlay[:, :, 1] = np.clip(overlay[:, :, 1] * (1.0 - 0.45 * norm), 0.0, 1.0)
    overlay[:, :, 2] = np.clip(overlay[:, :, 2] * (1.0 - 0.65 * norm), 0.0, 1.0)
    return overlay


def crop_from_box(image_array, box):
    x, y, w, h = box
    return image_array[y:y + h, x:x + w]


def replace_patch(query_image, replacement_image, query_box, source_box):
    updated = query_image.copy()
    xq, yq, wq, hq = query_box
    xs, ys, ws, hs = source_box
    source_patch = replacement_image[ys:ys + hs, xs:xs + ws]
    updated[yq:yq + hq, xq:xq + wq] = source_patch[:hq, :wq]
    return updated


def analysis_panel(image_array, scores, memory_bank, title):
    top = scores[0]
    nearest = top['nearest'][0]
    source_image = memory_bank['train_images'][nearest['metadata']['image_index']]
    heatmap = heatmap_from_scores(scores, image_array.shape)
    panels = [
        titled(np_to_pil(image_array), 'Query image'),
        titled(np_to_pil(overlay_heatmap(image_array, heatmap)), f'Anomaly map | {top["distance"]:.3f}'),
        titled(np_to_pil(crop_from_box(image_array, top['box'])), 'Top anomalous patch'),
        titled(np_to_pil(crop_from_box(source_image, nearest['metadata']['box'])), 'Nearest normal patch'),
    ]
    strip = montage(panels, cols=4)
    return titled(strip, title)

### Demo-specific image generation code

In [ ]:
def render_bottle(kind='clean', seed=0):
    rng = np.random.default_rng(seed)
    image = Image.new('RGB', (96, 128), (240, 236, 228))
    draw = ImageDraw.Draw(image)
    draw.rounded_rectangle((34, 18, 62, 112), radius=8, fill=(58, 144, 168), outline=(36, 58, 66), width=2)
    draw.rectangle((40, 8, 56, 22), fill=(96, 100, 104), outline=(36, 58, 66), width=2)
    draw.rectangle((38, 52, 58, 76), fill=(228, 235, 238))
    if kind == 'scratch':
        draw.line((58, 64, 72, 100), fill=(216, 92, 64), width=4)
    elif kind == 'stain':
        draw.ellipse((24, 72, 42, 92), fill=(74, 46, 28))
    elif kind == 'cap_shift':
        draw.rectangle((45, 8, 61, 22), fill=(96, 100, 104), outline=(36, 58, 66), width=2)
        draw.rectangle((40, 8, 44, 22), fill=(240, 236, 228))
    return np.clip(to_array(image) + rng.normal(0.0, 0.012, size=(128, 96, 3)), 0.0, 1.0)

### Build the synthetic nominal set and the query set

In [ ]:
train_images = [render_bottle('clean', seed=idx) for idx in range(10)]
memory_bank = build_memory_bank(train_images, patch=16, stride=8)
queries = {
    'clean control': render_bottle('clean', seed=20),
    'scratch anomaly': render_bottle('scratch', seed=21),
    'stain anomaly': render_bottle('stain', seed=22),
    'cap shift anomaly': render_bottle('cap_shift', seed=23),
}
maybe_display(montage([titled(image, label) for label, image in queries.items()], cols=4))

## Dataset and task definition
The notebook uses a synthetic bottle panel rather than a downloaded dataset. The nominal training set contains only clean bottles. The query set includes a clean control and three anomaly types:

- a scratch on the right edge;
- a dark stain outside the bottle;
- a shifted cap.

## Model and explanation methods
The model is a tiny PatchCore-style baseline built directly in the notebook.

- split each image into overlapping patches;
- represent each patch with mean and standard-deviation colour features;
- build a memory bank from nominal patches only;
- score each query patch by distance to the nearest nominal patch.

The explanation artefacts are the anomaly map, the top anomalous patch, and the nearest normal patch retrieved from the memory bank.

## Baseline result

In [ ]:
results = {name: score_image(image, memory_bank, top_k=3) for name, image in queries.items()}
for name, scores in results.items():
    print(name, 'top patch distance:', round(scores[0]['distance'], 3))
maybe_display(montage([
    analysis_panel(queries['clean control'], results['clean control'], memory_bank, 'Clean control'),
    analysis_panel(queries['scratch anomaly'], results['scratch anomaly'], memory_bank, 'Scratch anomaly'),
    analysis_panel(queries['stain anomaly'], results['stain anomaly'], memory_bank, 'Stain anomaly'),
], cols=1))

## Failure or pitfall

In [ ]:
maybe_display(analysis_panel(queries['cap shift anomaly'], results['cap shift anomaly'], memory_bank, 'Cap-shift anomaly with provenance'))

## Intervention

In [ ]:
scratch_scores = results['scratch anomaly']
top = scratch_scores[0]
nearest = top['nearest'][0]
source_image = memory_bank['train_images'][nearest['metadata']['image_index']]
patched = replace_patch(queries['scratch anomaly'], source_image, top['box'], nearest['metadata']['box'])
patched_scores = score_image(patched, memory_bank, top_k=3)
maybe_display(montage([
    titled(queries['scratch anomaly'], f'Original | {scratch_scores[0]["distance"]:.3f}'),
    titled(crop_from_box(source_image, nearest['metadata']['box']), 'Nearest normal patch'),
    titled(patched, f'Patched query | {patched_scores[0]["distance"]:.3f}'),
], cols=3))

## Re-test

In [ ]:
maybe_display(bar_chart(
    [scratch_scores[0]['distance'], patched_scores[0]['distance']],
    ['Original', 'Patched'],
    'Counterfactual patch replacement lowers the anomaly score',
))

## What we learned
- The explanation is richer than an anomaly heatmap alone.
- Nearest-normal provenance can be displayed directly in the notebook.
- A simple patch replacement counterfactual provides a lightweight verification path.

## Residual risks and next questions
- This uses simple colour-statistic patch features, not a deep feature extractor.
- Synthetic bottles are much easier than real industrial surfaces.
- A fuller demo would add stronger localisation metrics and more realistic nominal variation.